<a href="https://colab.research.google.com/github/ericalh04/DATA602TermProject/blob/main/kmdan_DATA602_TermProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Preliminary Processing and EDA

This data source is downloaded from the CDC and Census Bureau datsets. These were cleaned and processed in Excel and uploaded to our Google Drive. After connecting our Google Drive to Colab, we can access these data files saved to a Drive folder.

In [ ]:
# Connect Google drive to Colab

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Please note that in these datasets, Estimated Population (2000-2025) includes the year 2025. This will be important for our ML portion of the project.

The model for open population growth is:

Change in Population = Birth - Death + Immigration - Emigration

We can simplify this model as:
Change in Population = Natural Growth + Migratory Growth

First, we need to quantify Natural Growth and Migratory Growth between each year for each county. We will do this using a percent change.

In [ ]:
import pandas as pd

estimated_population_df = pd.read_excel('/content/drive/MyDrive/DATA602_ProjectData/clean_data/us_county_estpop_2000_2025.xlsx')
births_df = pd.read_csv('/content/drive/MyDrive/DATA602_ProjectData/us_county_births_2000_2024.csv')
netmigration_df = pd.read_csv('/content/drive/MyDrive/DATA602_ProjectData/clean_data/us_county_net_migration_2000_2024.csv')
mortality_df = pd.read_csv('/content/drive/MyDrive/DATA602_ProjectData/clean_data/us_county_mortality_2000_2024.csv')

The CDC does not report number of birth for counties with a population less than 100,000. So we can estimate Natural Growth by rearranging the Change in Population equation;

Natural Growth = Change in Population - Migration Growth.

Natural growth will be saved as us_county_natural_growth_2000_2024.csv

Please note that 2000 is listed as the estimated population as a baseline.

In [ ]:

import pandas as pd
import numpy as np

years = range(2001, 2025)
population_cols = [f"{year}_netnat" for year in years]

# format
col_names = ["FIPS", "County", "State"]
new_col = estimated_population_df[col_names + ["2000_estpop"]].copy()

# Calculate natural change
for year in years:
    current_year = f"{year}_estpop"
    previous_year = f"{year - 1}_estpop"
    migration_col = f"{year}_netmig"
    natural_col = f"{year}_naturalchange"

    current_population = pd.to_numeric(
        estimated_population_df[current_year],
        errors="coerce"
    )

    previous_population = pd.to_numeric(
        estimated_population_df[previous_year],
        errors="coerce"
    )

    migration_growth = pd.to_numeric(
        netmigration_df[migration_col],
        errors="coerce"
    )

    # Total population change
    population_change = current_population - previous_population

    # Natural change = population change - migration growth
    natural_change = population_change - migration_growth
    new_col[natural_col] = natural_change

new_col.to_csv(
    "us_county_natural_growth_2000_2024.csv",
    index=False
)

Next, we need to calculate the population change for each county from year to year. This is saved as us_county_population_change_2000_2025.csv

In [ ]:
import pandas as pd
import numpy as np

us_county_population_change = estimated_population_df[["FIPS", "County", "State"]].copy()

# 2000 as initial population
us_county_population_change["2000_estpop"] = (
    pd.to_numeric(
        estimated_population_df["2000_estpop"],
        errors="coerce"
    )
)

# Calculate year-to-year population change
for year in range(2001, 2026):

    current_year = f"{year}_estpop"
    previous_year = f"{year - 1}_estpop"
    change_year = f"{year}_popchange"

    current_population = (
        pd.to_numeric(
            estimated_population_df[current_year],
            errors="coerce"
        )
        -
        pd.to_numeric(
            estimated_population_df[previous_year],
            errors="coerce"
        )
    )
    us_county_population_change[change_year] = current_population

us_county_population_change.to_csv(
    "us_county_population_change_2000_2025.csv",
    index=False
)

We will now quantify the contribution of natural growth and migratory growth to the total population change for each county.

---

The calculation is:

Natural Contribution % = (Natural Change/Total Population Change) x 100


Migration Contribution % = (Migration Change/Total Population Change) x 100

In [ ]:
quantified_population_change = us_county_population_change[["FIPS", "County", "State"]].copy()

for year in range(2001, 2025):

    population_col = f"{year}_popchange"
    natural_growth_col = f"{year}_naturalchange"
    migratory_growth_col = f"{year}_netmig"

    natural_percent_col = f"{year}_natural%"
    migration_percent_col = f"{year}_migration%"

    total_population_change = pd.to_numeric(
        popchange_df[population_col],
        errors="coerce"
    )

    natural_change = pd.to_numeric(
        naturalchange_df[natural_growth_col],
        errors="coerce"
    )

    migration_change = pd.to_numeric(
        netmigration_df[migratory_growth_col],
        errors="coerce"
    )

    # Calculate contribution percentages
    natural_percent = (
        natural_change / total_population_change
    ) * 100

    migration_percent = (
        migration_change / total_population_change
    ) * 100

    # Save percentages
    quantified_population_change[natural_percent_col] = natural_percent
    quantified_population_change[migration_percent_col] = migration_percent

quantified_population_change.to_csv(
    "us_county_quantified_population_change_2000_2025.csv",
    index=False
)

For our hypothesis test, we need to combine the dataframes with net migration and natural change.

In [ ]:
# format
combined_growth_df = naturalchange_df[
    ["FIPS", "County", "State"]
].copy()

# natural growth and net migration for each year
for year in range(2001, 2025):
    natural_col = f"{year}_naturalchange"
    migration_col = f"{year}_netmig"

    combined_growth_df[natural_col] = pd.to_numeric(
        naturalchange_df[natural_col],
        errors="coerce"
    )

    combined_growth_df[migration_col] = pd.to_numeric(
        netmigration_df[migration_col],
        errors="coerce"
    )


combined_growth_df.to_csv(
    "us_county_HYPOTHESIS_TEST.csv",
    index=False
)


To perform the hypothesis test, we will transform the df into a wide format

In [ ]:
import pandas as pd

results = []

for year in range(2001, 2025):

    format = pd.DataFrame({
        "FIPS": combined_growth_df["FIPS"],
        "County": combined_growth_df["County"],
        "State": combined_growth_df["State"],
        "Year": year,

        "Natural_Growth": pd.to_numeric(
            combined_growth_df[f"{year}_naturalchange"],
            errors="coerce"
        ),

        "Net_Migration": pd.to_numeric(
            combined_growth_df[f"{year}_netmig"],
            errors="coerce"
        )
    })

    results.append(format)

growth_long_df = pd.concat(
    results,
    ignore_index=True
)

growth_long_df.to_csv(
    "us_county_growth_long_2001_2024.csv",
    index=False
)

len(growth_long_df)

75432